# TracIn: Finding Influential Training Examples

**Learning objectives:**
- Train a model with checkpoint saving for TracIn compatibility
- Use TracInCPFast to find top proponents and opponents for test predictions
- Compute self-influence to identify potentially mislabeled training examples
- Visualize influential example galleries and interpret findings

**Estimated time:** 15 minutes

**Model:** CNN trained on CIFAR-10 (torchvision)

**Method:** `TracInCPFast` from `captum.influence`

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from captum.influence import TracInCPFast

# Directories
os.makedirs('cifar10_checkpoints', exist_ok=True)
os.makedirs('cifar10_data', exist_ok=True)

CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

print(f"PyTorch: {torch.__version__}")
print(f"CIFAR-10 classes: {CIFAR10_CLASSES}")

## 1. Load CIFAR-10 and Introduce Mislabeled Examples

We load a subset of CIFAR-10 and deliberately mislabel some examples to test self-influence detection.

In [ ]:
transform_train = T.Compose([
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# Use a small subset for fast training (TracIn needs checkpoints)
train_full = CIFAR10(root='cifar10_data', train=True, download=True, transform=transform_train)
test_full  = CIFAR10(root='cifar10_data', train=False, download=True, transform=transform_test)

# Use 5000 training, 500 test for speed
np.random.seed(42)
train_idx = np.random.choice(len(train_full), 5000, replace=False)
test_idx  = np.random.choice(len(test_full),  500,  replace=False)

train_dataset = Subset(train_full, train_idx)
test_dataset  = Subset(test_full,  test_idx)

# Introduce 3% label noise (intentionally mislabeled)
MISLABELED_INDICES = train_idx[:int(0.03 * len(train_idx))]
print(f"Intentionally mislabeled: {len(MISLABELED_INDICES)} training examples")

# Create a custom dataset with label noise
from torch.utils.data import Dataset

class NoisySubset(Dataset):
    """CIFAR-10 subset with intentional label noise."""
    def __init__(self, dataset, noisy_subset_indices, noise_fraction=0.03, seed=0):
        self.dataset = dataset
        self.noisy_set = set(noisy_subset_indices)
        np.random.seed(seed)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        true_idx = self.dataset.indices[idx] if hasattr(self.dataset, 'indices') else idx
        if true_idx in self.noisy_set:
            # Flip label to a random wrong class
            wrong_label = (label + np.random.randint(1, 10)) % 10
            return img, wrong_label
        return img, label


noisy_train = NoisySubset(train_dataset, MISLABELED_INDICES)
train_loader = DataLoader(noisy_train, batch_size=128, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)

print(f"Training samples: {len(noisy_train)}")
print(f"Test samples:     {len(test_dataset)}")

## 2. Train CNN with Checkpoint Saving

TracIn requires checkpoints. We save them at 3 points during training.

In [ ]:
class SmallCNN(nn.Module):
    """Compact CNN for CIFAR-10. Compatible with TracInCPFast."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = SmallCNN()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
criterion = nn.CrossEntropyLoss()

checkpoint_config = []  # will hold (path, lr) tuples
checkpoint_epochs = [5, 12, 20]

print("Training CNN with checkpoint saving...")
for epoch in range(1, 21):
    model.train()
    total_loss = 0
    for imgs, labels in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()

    if epoch in checkpoint_epochs:
        ckpt_path = f'cifar10_checkpoints/model_epoch_{epoch:02d}.pt'
        torch.save(model.state_dict(), ckpt_path)
        checkpoint_config.append((ckpt_path, current_lr))
        print(f"  Epoch {epoch:2d}: loss={total_loss/len(train_loader):.3f}, lr={current_lr:.4f} → checkpoint saved")
    elif epoch % 5 == 0:
        print(f"  Epoch {epoch:2d}: loss={total_loss/len(train_loader):.3f}")

# Evaluate
model.eval()
correct = total = 0
with torch.no_grad():
    for imgs, labels in test_loader:
        preds = model(imgs).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += len(labels)
print(f"\nTest accuracy: {correct/total:.1%}")
print(f"Checkpoints saved: {checkpoint_config}")

## 3. Set Up TracInCPFast

In [ ]:
# TracInCPFast: uses last-layer gradients only (efficient approximation)
# The final linear layer is model.classifier[-1]

def load_checkpoint(model, path):
    model.load_state_dict(torch.load(path, map_location='cpu'))
    return model


# TracInCPFast requires the training dataset (not loader) for index lookup
tracin = TracInCPFast(
    model=model,
    train_dataset=noisy_train,
    checkpoints=checkpoint_config,      # list of (path, lr)
    checkpoints_load_func=load_checkpoint,
    loss_fn=nn.CrossEntropyLoss(reduction='none'),
    batch_size=64,
    vectorize=False,
)

print("TracInCPFast initialized.")
print(f"  Checkpoints: {len(checkpoint_config)}")
print(f"  Training examples: {len(noisy_train)}")

## 4. Find Proponents and Opponents for Test Predictions

In [ ]:
# Select test examples: one correct prediction, one wrong prediction
model.eval()
correct_idx = wrong_idx = None

with torch.no_grad():
    for i in range(min(100, len(test_dataset))):
        img, true_label = test_dataset[i]
        img_t = img.unsqueeze(0)
        pred = model(img_t).argmax(dim=1).item()
        if pred == true_label and correct_idx is None:
            correct_idx = i
        if pred != true_label and wrong_idx is None:
            wrong_idx = i
        if correct_idx is not None and wrong_idx is not None:
            break

# Get the test examples
test_examples = []
for idx, label in [(correct_idx, 'Correct'), (wrong_idx, 'Misclassified')]:
    img, true_label = test_dataset[idx]
    img_t = img.unsqueeze(0)
    with torch.no_grad():
        pred = model(img_t).argmax(dim=1).item()
    test_examples.append({
        'img': img,
        'true': true_label,
        'pred': pred,
        'label': label,
        'idx': idx,
    })
    print(f"  {label}: true={CIFAR10_CLASSES[true_label]}, pred={CIFAR10_CLASSES[pred]}")

In [ ]:
# Compute proponents and opponents for each test example
print("Computing TracIn influences (may take 1-2 minutes)...")

for ex in test_examples:
    img_t = ex['img'].unsqueeze(0)
    label_t = torch.tensor([ex['true']])

    # Top-5 proponents
    prop_inf, prop_idx = tracin.influence(
        inputs=(img_t, label_t),
        top_k=5,
        proponents=True,
    )

    # Top-5 opponents
    opp_inf, opp_idx = tracin.influence(
        inputs=(img_t, label_t),
        top_k=5,
        proponents=False,
    )

    ex['proponents'] = list(zip(prop_idx[0].tolist(), prop_inf[0].tolist()))
    ex['opponents']  = list(zip(opp_idx[0].tolist(), opp_inf[0].tolist()))
    print(f"  {ex['label']}: done")

print("TracIn complete.")

## 5. Visualize Proponents and Opponents

In [ ]:
def denormalize(tensor):
    """Undo CIFAR-10 normalization for display."""
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
    std  = torch.tensor([0.2023, 0.1994, 0.2010]).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


fig, big_axes = plt.subplots(2, 1, figsize=(18, 12))

for row, ex in enumerate(test_examples):
    # Create 11 subplots in this row: 1 test + 5 proponents + 5 opponents
    gs_inner = gridspec.GridSpecFromSubplotSpec(1, 12, subplot_spec=big_axes[row].get_subplotspec(),
                                                 wspace=0.05)
    big_axes[row].set_visible(False)

    # Helper to add subplot
    def add_img_subplot(spec_idx, img_tensor, title, title_color='black', border_color=None):
        ax = fig.add_subplot(gs_inner[spec_idx])
        ax.imshow(denormalize(img_tensor))
        ax.set_title(title, fontsize=7, color=title_color, fontweight='bold')
        ax.axis('off')
        if border_color:
            for spine in ax.spines.values():
                spine.set_edgecolor(border_color)
                spine.set_linewidth(3)
                spine.set_visible(True)
        return ax

    # Test image (col 0)
    is_correct = (ex['true'] == ex['pred'])
    border = '#4daf4a' if is_correct else '#e41a1c'
    add_img_subplot(
        0, ex['img'],
        f"TEST\n{CIFAR10_CLASSES[ex['true']]}\n({'✓' if is_correct else '✗ '+CIFAR10_CLASSES[ex['pred']]})",
        title_color=border, border_color=border
    )

    # Divider column (col 1)
    ax_div = fig.add_subplot(gs_inner[1])
    ax_div.axis('off')
    ax_div.text(0.5, 0.5, 'P\nR\nO\nP', ha='center', va='center',
                color='#d73027', fontsize=9, fontweight='bold')

    # Proponents (cols 2-6)
    for i, (train_i, inf_val) in enumerate(ex['proponents'][:5]):
        train_img, train_label = noisy_train[train_i]
        is_mislabeled = (train_idx[train_i] in set(MISLABELED_INDICES))
        add_img_subplot(
            i + 2, train_img,
            f"{CIFAR10_CLASSES[train_label]}\n{inf_val:+.3f}{'⚠️' if is_mislabeled else ''}",
            title_color='#d73027'
        )

    # Divider
    ax_div2 = fig.add_subplot(gs_inner[7])
    ax_div2.axis('off')
    ax_div2.text(0.5, 0.5, 'O\nP\nP', ha='center', va='center',
                 color='#4575b4', fontsize=9, fontweight='bold')

    # Opponents (cols 8-12)
    for i, (train_i, inf_val) in enumerate(ex['opponents'][:4]):
        train_img, train_label = noisy_train[train_i]
        add_img_subplot(
            i + 8, train_img,
            f"{CIFAR10_CLASSES[train_label]}\n{inf_val:+.3f}",
            title_color='#4575b4'
        )

    # Row label
    big_axes[row].set_ylabel(
        f"{'Correct' if is_correct else 'Wrong'}\nprediction",
        fontsize=11, fontweight='bold',
        color='#4daf4a' if is_correct else '#e41a1c'
    )

plt.suptitle('TracIn: Proponents (red) and Opponents (blue)\nfor Correct and Misclassified Predictions',
             fontsize=13, fontweight='bold')
plt.savefig('tracin_proponents_opponents.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: tracin_proponents_opponents.png")

## 6. Self-Influence: Detecting Mislabeled Examples

In [ ]:
print("Computing self-influence for all training examples...")
print("(This computes ||∇L(z)||² at each checkpoint)")

# Compute self-influence — measures how much each example influenced its own training
self_influences = tracin.self_influence(
    inputs=DataLoader(noisy_train, batch_size=64, shuffle=False),
    show_progress=True,
)

print(f"\nSelf-influence computed for {len(self_influences)} training examples.")
print(f"Range: [{self_influences.min().item():.4f}, {self_influences.max().item():.4f}]")
print(f"Mean:  {self_influences.mean().item():.4f}")

In [ ]:
# Analyze: do mislabeled examples have higher self-influence?
# Build a lookup: which training subset indices are mislabeled
mislabeled_set_local = set(range(len(MISLABELED_INDICES)))  # first 3% are mislabeled

si_mislabeled = self_influences[[i for i in range(len(self_influences)) if i in mislabeled_set_local]]
si_clean      = self_influences[[i for i in range(len(self_influences)) if i not in mislabeled_set_local]]

print(f"Mislabeled self-influence: mean={si_mislabeled.mean():.4f}, std={si_mislabeled.std():.4f}")
print(f"Clean self-influence:      mean={si_clean.mean():.4f}, std={si_clean.std():.4f}")

# Top-20 highest self-influence examples
top_si_idx = self_influences.argsort(descending=True)[:20]
n_mislabeled_in_top20 = sum(1 for idx in top_si_idx if idx.item() in mislabeled_set_local)
print(f"\nMislabeled examples in top-20 self-influence: {n_mislabeled_in_top20}/20")
print(f"(Expected by chance: {int(20 * len(mislabeled_set_local)/len(noisy_train))}")
print("Self-influence successfully identifies mislabeled examples!" if n_mislabeled_in_top20 > 5 else
      "Low detection — consider using more checkpoints or longer training.")

In [ ]:
# Visualize self-influence distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(si_clean.numpy(), bins=50, alpha=0.7, label='Clean labels', color='#4daf4a', density=True)
ax.hist(si_mislabeled.numpy(), bins=20, alpha=0.7, label='Mislabeled', color='#e41a1c', density=True)
ax.set_xlabel('Self-influence score')
ax.set_ylabel('Density')
ax.set_title('Self-Influence Distribution\nMislabeled examples cluster at higher values',
             fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# Precision-recall curve: self-influence as mislabel detector
from sklearn.metrics import precision_recall_curve, average_precision_score

labels_binary = np.array([1 if i in mislabeled_set_local else 0
                           for i in range(len(self_influences))])
si_scores = self_influences.numpy()

precision, recall, thresholds = precision_recall_curve(labels_binary, si_scores)
ap = average_precision_score(labels_binary, si_scores)

ax = axes[1]
ax.plot(recall, precision, 'b-', linewidth=2)
ax.fill_between(recall, precision, alpha=0.2, color='blue')
ax.axhline(labels_binary.mean(), color='gray', linestyle='--',
           label=f'Random baseline ({labels_binary.mean():.2%})')
ax.set_xlabel('Recall (fraction of mislabeled found)')
ax.set_ylabel('Precision (fraction of flagged that are mislabeled)')
ax.set_title(f'Precision-Recall: Self-Influence as Mislabel Detector\nAP={ap:.3f}',
             fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle('TracIn Self-Influence: Mislabeled Example Detection',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('tracin_self_influence.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved: tracin_self_influence.png")
print(f"Average Precision: {ap:.3f}  (1.0 = perfect, random = {labels_binary.mean():.3f})")

## 7. Visualize High Self-Influence Examples

In [ ]:
# Show top-20 highest self-influence examples
top20_idx = self_influences.argsort(descending=True)[:20]

fig, axes = plt.subplots(4, 5, figsize=(15, 12))

for i, train_i in enumerate(top20_idx):
    ax = axes[i // 5, i % 5]
    img, label = noisy_train[train_i.item()]
    si_val = self_influences[train_i].item()
    is_mislabeled = train_i.item() in mislabeled_set_local

    ax.imshow(denormalize(img))
    ax.set_title(
        f"{CIFAR10_CLASSES[label]}\nSI={si_val:.3f}\n{'⚠ MISLABELED' if is_mislabeled else 'clean'}",
        fontsize=8,
        color='#e41a1c' if is_mislabeled else 'black',
        fontweight='bold' if is_mislabeled else 'normal'
    )
    ax.axis('off')

    # Red border for mislabeled
    if is_mislabeled:
        for spine in ax.spines.values():
            spine.set_edgecolor('#e41a1c')
            spine.set_linewidth(3)
            spine.set_visible(True)

plt.suptitle('Top-20 Highest Self-Influence Training Examples\n(Red border = intentionally mislabeled)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('tracin_high_self_influence.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: tracin_high_self_influence.png")

## Self-Check Questions

1. **Checkpoint count:** We used 3 checkpoints. How would increasing to 10 checkpoints affect: (a) TracIn accuracy, (b) computation time?

2. **Proponents vs. similarity:** A proponent is the training example with the highest TracIn score, not the most visually similar image. Can you think of a case where the top proponent is visually different from the test image but still has high TracIn influence?

3. **Precision-recall interpretation:** The AP for self-influence as a mislabel detector is your computed value. What would AP=1.0 mean, and is it achievable in practice?

4. **Learning rate weighting:** TracIn weights contributions by learning rate $\eta_t$. If we used a constant learning rate throughout training, how would this affect which checkpoints matter most?

**Exercise:** Modify the training loop to save 6 checkpoints (every 3-4 epochs) instead of 3. Rerun TracIn and compare the self-influence AP score. Does more checkpoints improve mislabeled example detection?